## Layer 1

In [0]:
# This code-blocks needs to run regardless of execution starting point
# as it sets up the S3 paths and imports the necessary functions.
from pyspark.sql import functions as F

bucket = "s3://my-de-case-study-datalake/"
assignment_bucket = "s3a://merkle-de-interview-case-study/de/"

In [0]:

# If running from scratch, run theis code-block to fetch the raw data from the assignment S3 bucket.

item_df = spark.read.csv(
    "s3a://merkle-de-interview-case-study/de/item.csv",
    header=True,
    inferSchema=False,
    multiLine=False,
)

# do the same for the 2nd file

event_df = spark.read.csv(
    "s3a://merkle-de-interview-case-study/de/event.csv",
    header=True,
    inferSchema=False,
    multiLine=False,
    quote='"',  # treat double quotes as enclosing quotes
    escape='"',
)
# Save to local bronze layer
event_df.write.format("delta").mode("overwrite").save(f"{bucket}bronze/event_raw")
item_df.write.format("delta").mode("overwrite").save(f"{bucket}bronze/item_raw")

# If starting the notebook execution from this point, to avoid wasting resources, un-comment the below code-block to fetch the already loaded tables from the bronze layer.

#### If starting the notebook execution from this point, to avoid wasting resources, un-comment the below code-block to fetch the already loaded tables from the bronze layer.

In [0]:
# item_df = spark.read.format("delta").load(f"{bucket}bronze/item_raw")
# event_df = spark.read.format("delta").load(f"{bucket}bronze/event_raw")

## Layer 2

In [ ]:
# Helper functions
from pyspark.sql import functions as F, DataFrame
from pyspark.sql.functions import max as spark_max
from pyspark.sql.types import DecimalType


def check_column_has_unique_values(df, column_name):
    """
    Checks if all values in the specified column of a DataFrame are unique.
    Displays and raises an error if any duplicate values are found, otherwise prints a confirmation message.

    Parameters:
        df: pyspark.sql.DataFrame
            The DataFrame to check.
        column_name: str
            The name of the column to validate for uniqueness.
    """
    duplicates = (
        df.groupBy(F.col(column_name))
        .count()
        .filter(F.col("count") > 1)
    )
    if not duplicates.isEmpty():
        raise ValueError(
            f"Duplicate values found in column '{column_name}'. Workflow stopped."
        )



# Check for null category rows and halt workflow if any are found
def check_column_has_null_values(df, column_name):
    null_values_row = df.filter(F.col(column_name).isNull())
    if not null_values_row.isEmpty():
        raise ValueError(f"Null value(s) found in column '{column_name}'. Workflow stopped.")



def check_column_datetime_format(
    df, column_name, datetime_regex=r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
):
    """
    Checks if all values in the specified column of a DataFrame match the given datetime format regex.
    Displays and raises an error if any invalid values are found, otherwise prints a confirmation message.

    Parameters:
        df: pyspark.sql.DataFrame
            The DataFrame to check.
        column_name: str
            The name of the column to validate.
        datetime_regex: str, optional
            The regex pattern for the datetime format (default: yyyy-MM-dd HH:mm:ss).
    """
    invalid_dates = df.filter(~F.col(column_name).rlike(datetime_regex))
    if not invalid_dates.isEmpty():
        display(invalid_dates)
        raise ValueError(
            f"Invalid date format found in column '{column_name}'. Workflow stopped."
        )
    print(f"All values in column '{column_name}' have valid datetime format.")


def check_column_is_integer(df, column_name):
    """
    Checks if all values in the specified column of a DataFrame are integers (no decimal part).
    Displays and raises an error if any non-integer values are found, otherwise prints a confirmation message.

    Parameters:
        df: pyspark.sql.DataFrame
            The DataFrame to check.
        column_name: str
            The name of the column to validate for integer-ness.
    """
    col_with_decimal = df.filter((F.col(column_name).cast("float") % 1) != 0)
    if not col_with_decimal.isEmpty():
        display(col_with_decimal)
        raise ValueError(
            f"Non-integer values found in column '{column_name}'. Workflow stopped."
        )


def cast_columns_to_decimal_dynamic(
    df: DataFrame, columns: list, overprovision_digits: int = 0
) -> DataFrame:
    """
    Dynamically compute max precision and scale for multiple numeric columns
    and cast them to DecimalType without rounding.

    Parameters:
        df: Input Spark DataFrame
        columns: List of numeric column names to cast
        overprovision_digits: Extra digits to add to precision for future-proofing
    Returns:
        DataFrame with cast DecimalType columns
    """
    df_final = df
    for col_name in columns:
        tmp_col = f"__tmp_str_{col_name}"
        df_tmp = df_final.withColumn(tmp_col, F.col(col_name).cast("string"))
        split_col = F.split(F.col(tmp_col), r"\.")
        df_parts = df_tmp.withColumn(
            f"__int_len_{col_name}", F.length(split_col.getItem(0))
        ).withColumn(
            f"__frac_len_{col_name}",
            F.when(
                split_col.getItem(1).isNotNull(), F.length(split_col.getItem(1))
            ).otherwise(0),
        )
        max_lengths = df_parts.agg(
            spark_max(f"__int_len_{col_name}").alias("max_int_len"),
            spark_max(f"__frac_len_{col_name}").alias("max_frac_len"),
        ).collect()[0]
        precision = (
            max_lengths["max_int_len"]
            + max_lengths["max_frac_len"]
            + overprovision_digits
        )
        scale = max_lengths["max_frac_len"]

        print(f"[INFO] Column '{col_name}': precision={precision}, scale={scale}")
        
        df_final = df_final.withColumn(
            col_name, F.col(col_name).cast(DecimalType(precision, scale))
        )
        df_final = df_final.drop(tmp_col)
    return df_final

### Attribute type validation
Reflecting a production ready flow (minus the logger implementation)

In [0]:
# Check for duplicates and stop workflow if id is not unique
from pyspark.sql import functions as F

dupes = item_df.groupBy("id").agg(F.count("*").alias("cnt")).filter(F.col("cnt") > 1)
dupe_count = dupes.count()
if dupe_count > 0:
    raise ValueError(
        f"id is not unique in item_df! Found {dupe_count} duplicate id(s). Workflow stopped."
    )

In [0]:
null_check_columns = {"item": ["category", "id", "price"], 
                      "event": ["event_time", "event_payload"]}
unique_check_columns = {"item": ["id"], "event": ["event_id"]}

# A try-catch block could be used to handle errors more gracefully in the context of a larger workflow
for col in unique_check_columns["item"]:
    check_column_has_unique_values(item_df, col)
for col in unique_check_columns["event"]:
    check_column_has_unique_values(event_df, col)
    
for col in null_check_columns["item"]:
    check_column_has_null_values(item_df, col)
for col in null_check_columns["event"]:
    check_column_has_null_values(event_df, col)

check_column_is_integer(item_df, "id")
check_column_is_integer(event_df, "user_id")
check_column_datetime_format(item_df, "created_at")
check_column_datetime_format(event_df, "event_time")

### Attribute type casting

#### Item DataFrame transformations

In [0]:
from pyspark.sql import functions as F

# cast basic columns that don’t require dynamic DecimalType
item_df_parsed = (
    item_df.withColumn("adjective", F.col("adjective").cast("string"))
    .withColumn("category", F.col("category").cast("string"))
    .withColumn("created_at", F.to_timestamp("created_at", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("id", F.col("id").cast("int"))
    .withColumn("modifier", F.col("modifier").cast("string"))
    .withColumn("name", F.col("name").cast("string"))
)

# cast price dynamically
item_df_parsed = cast_columns_to_decimal_dynamic(
    item_df_parsed,
    columns=["price"],
    overprovision_digits=2,  # optional buffer for future-proofing
)

#### Event DataFrame transformations

##### Expand the nested struct of event.payload

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

json_schema = StructType(
    [
        StructField("event_name", StringType(), True),
        StructField("platform", StringType(), True),
        StructField("parameter_name", StringType(), True),
        StructField("parameter_value", StringType(), True),
    ]
)

In [0]:
event_df_parsed = (
    event_df.withColumn(
        "event_payload_struct", F.from_json(F.col("`event.payload`"), json_schema)
    )
    .select(
        *[c for c in event_df.columns if c != "event.payload"],
        F.col("event_payload_struct.event_name").alias("payload_event_name"),
        F.col("event_payload_struct.platform").alias("payload_platform"),
        F.col("event_payload_struct.parameter_name").alias("payload_parameter_name"),
        # The parameter value can be a string or a number so I leave it as a string
        F.col("event_payload_struct.parameter_value").alias("payload_parameter_value")
    )
    .withColumn("user_id", F.col("user_id").cast("float").cast("int"))
    .withColumn("event_time", F.to_timestamp("event_time", "yyyy-MM-dd HH:mm:ss"))
)

### Load the dataframes into the silver layer with no removals

In [0]:
# Only the year partition is needed for this assignment, so I removed the month. 
# Can be added back if needed.
events_df_partitioned = event_df_parsed.withColumn(
    "year", F.year(F.col("event_time")))
# .withColumn("month", F.month(F.col("event_time")))

events_df_partitioned.write.format("delta").mode("overwrite").partitionBy(
    "year", "payload_event_name"
).save(f"{bucket}silver/event_fact")

In [0]:
# The assignment does not specify price buckets so I don't use them in the partitioning.
# Kept the category partitioning as the assignment asked for meaningful partitions, 
# I assume not just to the basic extend of the required datamart
item_df_parsed.write.mode("overwrite").format("delta").partitionBy("category").save(
    f"{bucket}silver/item_clean"
)

## Useful partitions:
- Period of the year (vacations, Christmas, Black Friday) might be when the most expensive ones get sold -> don't do sales
- Time of the day (working hours, past midnight) might be when the cheapest ones get sold -> increase their price
- platforms
- product adjective, modifier, category

#  Layer 3: Datamart

### If starting the notebook execution from this point, to avoid wasting resources, un-comment the below code-block to fetch the already loaded tables from the silver layer.

In [0]:
item_df_parsed = spark.read.format("delta").load(f"{bucket}/silver/item_clean")
event_df_parsed = spark.read.format("delta").load(f"{bucket}/silver/event_fact")

### Total number of item views in a particular year.

In [0]:
from pyspark.sql import functions as F, Window


views_df = (
    event_df_parsed.filter(
        (F.col("payload_event_name") == "view_item")
        & (F.col("payload_parameter_name") == "item_id")
    )  # filter further for the response with the item id
    .withColumnRenamed("payload_platform", "platform")
    .withColumnRenamed("payload_event_name", "event_name")
)

# Join with item dimension
views_with_items = views_df.join(
    item_df_parsed, views_df["payload_parameter_value"] == item_df_parsed["id"], "inner"
)

# Aggregate: total views + most used platform
agg_df = (
    views_with_items.groupBy("id", "name", "year")
    .agg(
        F.count("*").alias("total_views"),
        F.expr("mode() within group (order by platform)").alias(
            "most_used_platform"
        ),  # for Spark SQL 3.x
    )
    .orderBy("id", "total_views")
)

# Add ranking by total views per year
window_rank = Window.partitionBy("year").orderBy(F.desc("total_views"))
top_item_df = agg_df.withColumn("rank", F.dense_rank().over(window_rank))

In [0]:
# Save to Gold layer
top_item_df.write.mode("overwrite").format("delta").save(f"{bucket}gold_layer/top_item")